# SynthID-Bypass v2 on Kaggle
Runs the v2 ComfyUI workflow from [cebeuq/Synthid-Bypass](https://github.com/cebeuq/Synthid-Bypass) headlessly
(Qwen-Image GGUF redraw + Z-Image Turbo face pass). Separate from the v1 notebook: own ComfyUI (v0.9.2), own nodes, own models.

Settings: **Accelerator: GPU**, **Internet: On**. Needs ~38 GB of scratch disk (models ~30 GB). Details: `docs/KAGGLE_V2.md`.

In [ ]:
# 1. Pick the scratch dir with the most free space (models do not fit in /kaggle/working).
import os, shutil
best, best_free = None, -1
for c in ["/kaggle/temp", "/kaggle/tmp", "/tmp"]:
    try:
        os.makedirs(c, exist_ok=True)
        free = shutil.disk_usage(c).free / 1e9
        print(f"{c}: {free:.1f} GB free")
        if free > best_free: best, best_free = c, free
    except Exception as e:
        print(f"{c}: unusable ({e})")
SCRATCH = os.path.join(best, "synth_v2")
os.makedirs(SCRATCH, exist_ok=True)
assert best_free >= 38, f"Only {best_free:.0f} GB free in {best}; need ~38 GB"
os.environ["SYNTH_V2_DIR"] = SCRATCH
os.environ["HF_HOME"] = f"{SCRATCH}/hf_home"
os.environ["PIP_CACHE_DIR"] = f"{SCRATCH}/pip_cache"
os.environ["TORCH_HOME"] = f"{SCRATCH}/torch_home"
print("Using scratch:", SCRATCH)

In [ ]:
# 2. Get the runner code (small, fine in /kaggle/working).
import subprocess, sys
REPO = "https://github.com/de3sec/synth-id-remover"
APP = "/kaggle/working/synth-id-remover"
if not os.path.exists(APP):
    subprocess.run(["git", "clone", REPO, APP], check=True)
else:
    subprocess.run(["git", "remote", "set-url", "origin", REPO], cwd=APP, check=True)
    subprocess.run(["git", "fetch", "-q", "origin"], cwd=APP, check=True)
    subprocess.run(["git", "reset", "-q", "--hard", "origin/main"], cwd=APP, check=True)
assert os.path.exists(f"{APP}/v2/synthid_v2.py"), "v2/ is not on GitHub yet: commit and push it, then re-run this cell"
if f"{APP}/v2" not in sys.path: sys.path.insert(0, f"{APP}/v2")

In [ ]:
# 3. Setup: pinned ComfyUI + custom nodes + pip deps + ~30 GB of models. Takes a while; safe to re-run.
!pip install -q huggingface_hub hf_transfer
import importlib; importlib.invalidate_caches()
if "synthid_v2" in sys.modules:  # re-running must not keep stale code or an old ComfyUI process
    sys.modules["synthid_v2"].stop_server(); importlib.reload(sys.modules["synthid_v2"])
import synthid_v2 as v2
v2.setup()
!du -sh {SCRATCH}/ComfyUI/models/* ; df -h {SCRATCH}

In [ ]:
# 4. Dry run (no image needed): starts ComfyUI, checks every custom node loaded, converts the workflow.
# If a node is missing, the error lists the import failures from ComfyUI's log.
v2.start_server()
prompt = v2.build_prompt()
print(len(prompt), "nodes:", sorted({n["class_type"] for n in prompt.values()}))

In [ ]:
# 5a. Run on an image (add it via "Add Input" as a dataset, or put a path here)
from PIL import Image
IMG = "/kaggle/input/YOUR_DATASET/image.png"   # <-- change me
ADAPTIVE_LEVEL = 6      # 1-10. The workflow's own note: use 7-8 for higher-resolution inputs
MAX_MEGAPIXELS = None   # e.g. 2.0 to downscale big inputs first (saves time/VRAM on a T4)
out = v2.process(Image.open(IMG), adaptive_level=ADAPTIVE_LEVEL, max_megapixels=MAX_MEGAPIXELS)
out.save("/kaggle/working/output_v2.png")
out

In [ ]:
# 5b. Or launch a Gradio UI with a public share link
v2.launch_ui(share=True)

**Notes**
- Scratch is wiped when the session ends, so setup + downloads repeat each session (~30 GB).
- The Qwen-Image GGUF is ~13 GB, so on a 15 GB T4 ComfyUI offloads part of it to RAM: expect minutes per image, more for large inputs.
- `v2.process(..., timeout=3600)` is the wait limit; if it fails, the error shows the failing node and ComfyUI's log tail.